In [35]:
import numpy as np
import pandas as pd

In [36]:
df = pd.read_json("../data/processed/steam_games_processed.json")
df.head()

,appid,name,required_age,price,short_description,windows,mac,linux,achievements,supported_languages,...,genres,tags,release_month,market_competition,target_num,screenshots_count,has_header_image,has_website,has_support_url,has_support_email
0,496350,Supipara - Chapter 1 Spring Has Come!,0,5.24,"Spring has come, and our protagonist, Yukinari...",1,0,0,0,[English],...,[Adventure],"[Adventure, Visual Novel, Anime, Cute]",7,4128,0,10,1,1,1,1
1,1034400,Mystery Solitaire The Black Raven,0,4.99,Discover an entrancing and spectacular world!,1,1,0,0,"[English, German, Russian, French]",...,[Casual],"[Casual, Card Game, Solitaire, Puzzle, Hidden ...",5,7229,0,7,1,1,1,1
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,0,8.99,Yuha! I'll start the broadcast! Hakko's extrem...,1,0,0,19,[Korean],...,"[Casual, Indie, Simulation]",[],10,15555,0,7,1,0,0,1
3,3631080,Maze Quest VR,0,4.99,Its not just a Maze; its a Quest! Enter the ca...,1,0,0,0,[English],...,"[Action, Early Access]",[],4,14680,0,6,1,1,1,1
4,1654170,Agony VR,0,13.99,Agony VR is a first-person survival horror gam...,1,0,0,0,"[German, Spanish - Spain, Turkish, Russian, It...",...,"[Action, Adventure]",[],4,11675,0,5,1,0,1,1


In [37]:
df.drop(columns=['appid', 'name'], inplace=True)

/tmp/ipykernel_35882/3430216381.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=['appid', 'name'], inplace=True)


In [38]:
df.columns

Index(['required_age', 'price', 'short_description', 'windows', 'mac', 'linux',
       'achievements', 'supported_languages', 'full_audio_languages',
       'developers', 'publishers', 'categories', 'genres', 'tags',
       'release_month', 'market_competition', 'target_num',
       'screenshots_count', 'has_header_image', 'has_website',
       'has_support_url', 'has_support_email'],
      dtype='str')

Dataset split

In [39]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target_num'])

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42, stratify=df_full_train['target_num'])

In [40]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

Preparing the target variable

In [41]:
y_train = df_train['target_num']
y_val = df_val['target_num']
y_test = df_test['target_num']

In [42]:
del df_train['target_num']
del df_val['target_num']
del df_test['target_num']

Encoding

In [43]:
for df_ in [df_train, df_val, df_test]:
    df_['primary_developer'] = df_['developers'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown'
    )
    df_['primary_publisher'] = df_['publishers'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown'
    )

    df_['short_description'] = df_['short_description'].fillna('').astype(str)

MultiLabel for genres, categories, tags, languages

In [52]:
from sklearn.preprocessing import MultiLabelBinarizer


def encode_multilabel_feature(train_series, val_series, test_series, prefix):
    mlb = MultiLabelBinarizer()
    
    mlb.fit(train_series)

    train_encoded = pd.DataFrame(mlb.transform(train_series), columns=[f"{prefix}_{c}" for c in mlb.classes_])
    val_encoded = pd.DataFrame(mlb.transform(val_series), columns=[f"{prefix}_{c}" for c in mlb.classes_])
    test_encoded = pd.DataFrame(mlb.transform(test_series), columns=[f"{prefix}_{c}" for c in mlb.classes_])
    
    return train_encoded, val_encoded, test_encoded

In [53]:
genres_train, genres_val, genres_test = encode_multilabel_feature(df_train['genres'], df_val['genres'], df_test['genres'], 'genre')

cats_train, cats_val, cats_test = encode_multilabel_feature(df_train['categories'], df_val['categories'], df_test['categories'], 'cat')

tags_train, tags_val, tags_test = encode_multilabel_feature(df_train['tags'], df_val['tags'], df_test['tags'], 'tag')

audio_langs_train, audio_langs_val, audio_langs_test = encode_multilabel_feature(
    df_train['full_audio_languages'], 
    df_val['full_audio_languages'], 
    df_test['full_audio_languages'], 
    'audio_lang')

langs_train, langs_val, langs_test = encode_multilabel_feature(
    df_train['supported_languages'], 
    df_val['supported_languages'], 
    df_test['supported_languages'], 
    'lang')

/home/sn/Projects/steam-success-predictor/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['Batman', 'Football', 'Tile-Matching'] will be ignored
  warnings.warn(


Target encoding for developers and publishers

In [54]:
df_train.columns

Index(['required_age', 'price', 'short_description', 'windows', 'mac', 'linux',
       'achievements', 'supported_languages', 'full_audio_languages',
       'developers', 'publishers', 'categories', 'genres', 'tags',
       'release_month', 'market_competition', 'screenshots_count',
       'has_header_image', 'has_website', 'has_support_url',
       'has_support_email', 'primary_developer', 'primary_publisher'],
      dtype='str')

In [48]:
from category_encoders import TargetEncoder

te = TargetEncoder(cols=['primary_developer', 'primary_publisher'], smoothing=10.0)

te.fit(df_train[['primary_developer', 'primary_publisher']], y_train)

dev_pub_train = te.transform(df_train[['primary_developer', 'primary_publisher']]).add_suffix('_te')
dev_pub_val = te.transform(df_val[['primary_developer', 'primary_publisher']]).add_suffix('_te')
dev_pub_test = te.transform(df_test[['primary_developer', 'primary_publisher']]).add_suffix('_te')

NLP for short description

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300, stop_words='english')

tfidf.fit(df_train['short_description'])

tfidf_cols = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]

tfidf_train = pd.DataFrame(tfidf.transform(df_train['short_description']).toarray(), columns=tfidf_cols)
tfidf_val = pd.DataFrame(tfidf.transform(df_val['short_description']).toarray(), columns=tfidf_cols)
tfidf_test = pd.DataFrame(tfidf.transform(df_test['short_description']).toarray(), columns=tfidf_cols)

All together

In [50]:
df_train.columns

Index(['required_age', 'price', 'short_description', 'windows', 'mac', 'linux',
       'achievements', 'supported_languages', 'full_audio_languages',
       'developers', 'publishers', 'categories', 'genres', 'tags',
       'release_month', 'market_competition', 'screenshots_count',
       'has_header_image', 'has_website', 'has_support_url',
       'has_support_email', 'primary_developer', 'primary_publisher'],
      dtype='str')

In [ ]:
base_features = [
    'required_age', 
    'price', 
    'windows', 
    'mac', 
    'linux',
    'achievements', 
    'release_month', 
    'market_competition', 
    'screenshots_count',
    'has_header_image', 
    'has_website', 
    'has_support_url',
    'has_support_email']

X_train_final = pd.concat([
    df_train[base_features], genres_train, cats_train, tags_train, langs_train, dev_pub_train, tfidf_train
], axis=1)

X_val_final = pd.concat([
    df_val[base_features], genres_val, cats_val, tags_val, langs_val, dev_pub_val, tfidf_val
], axis=1)

X_test_final = pd.concat([
    df_test[base_features], genres_test, cats_test, tags_test, langs_test, dev_pub_test, tfidf_test
], axis=1)

print(f"Train shape: {X_train_final.shape}")
print(f"Val shape:   {X_val_final.shape}")
print(f"Test shape:  {X_test_final.shape}")

Train shape: (61926, 969)
Val shape:   (20642, 969)
Test shape:  (20642, 969)
